# Fusing Depth Data

## Overview

Most cryptocurrency exchanges do not deliver true tick-by-tick Level-2 data. Instead, they provide conflated feeds in which individual order-book updates are aggregated over short intervals. For example, Binance Futures’ `depth@0ms` stream is still aggregated: You can confirm that its best-bid-offer (BBO) values update less frequently than those in the `bookTicker` stream, which captures every BBO change. Other venues state similar limitations explicitly—Bybit, for instance, publishes the Level 1 data (BBO) every 10ms, the Level 50 data every 20ms, and the Level 200 data every 100ms.

To generate accurate fill simulations and realistic backtesting results, you must therefore fuse multiple depth streams into a single feed that preserves the highest possible update frequency and granularity.

Let’s see Binance Futures as our example.

## Runnable Tardis Test

This notebook executes the corresponding experiment through the shared
`tutorial_reproduction` runner. It uses existing Tardis files only and
does not require a Tardis API key or download data.

Defaults:

- amdserver: `/home/molly/data/tardis/binance-futures`, `2025-08-01`
- Mac: `~/Documents/tardis`, `2025-01-01`
- Window: `300` seconds

Optional environment overrides:

- `HFTBACKTEST_TARDIS_ROOT`
- `HFTBACKTEST_TARDIS_DATE`
- `HFTBACKTEST_NOTEBOOK_SECONDS`
- `HFTBACKTEST_NOTEBOOK_OUTPUT`

Active experiment: `Fusing Depth Data.ipynb` (`fusing_depth_data`).

In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
search_roots = []
for base in (cwd, *cwd.parents):
    search_roots.extend((base, base / 'examples'))
examples_root = next(
    path for path in search_roots
    if (path / 'tutorial_reproduction').is_dir()
)
if str(examples_root) not in sys.path:
    sys.path.insert(0, str(examples_root))

from tutorial_reproduction.notebook_support import (
    context_dict,
    notebook_context,
    run_notebook_experiment,
)

In [ ]:
context = notebook_context()
context_dict(context)

In [ ]:
manifest = run_notebook_experiment('fusing_depth_data', context)
manifest['result']

In [ ]:
assert manifest['result']['status'] != 'failed'
print('notebook:', manifest['notebook'])
print('status:', manifest['result']['status'])
print('output:', context.output_root)

## Original Tutorial Reference

The original tutorial narrative and code are retained below for comparison.
Original code cells are rendered as non-executing references so that
`Run All` remains reproducible with the configured Tardis dataset.

## Data Preparation

Use the backtester to replay the data, get the BBO values from the Level-2 depth feed to compare it with the BBO obtained from the book ticker stream.

## Comparing BBO updates: Level-2 (depth@0ms) Stream vs bookTicker Stream

The `bookTicker` stream delivers updates more often and leads the Level-2 feed by a small margin.

You’ll notice that the `bookTicker` stream delivers updates far more frequently—especially when you factor in changes to both price and quantity.

## Fusing Multiple Depth Feeds

To obtain the most frequent and fine-grained market depth updates, it is necessary to combine the depth feed with the book ticker feed. The [`FuseMarketDepth`](https://hftbacktest.readthedocs.io/en/latest/reference/data_utilities.html#hftbacktest.binding.FuseMarketDepth) utility helps fuse multiple depth update streams into a single order book view. HftBacktest includes a fused converter function [`convert_fuse`](https://hftbacktest.readthedocs.io/en/latest/reference/hftbacktest.data.utils.tardis.html#hftbacktest.data.utils.tardis.convert_fuse) for Tardis data.
 
<div class="alert alert-info">

**Note: Handling Timestamp Inconsistencies**  
When fusing multiple depth feeds, it's possible that an event with a later exchange timestamp may be received before an event with an earlier timestamp from another feed. Accurately reconstructing the order book in such cases would require building a separate order book for each feed and then combining them.

However, this utility builds only a single consolidated order book, where:
- Updates are processed in order of local receipt time.
- The price-level information are updated based on the exchange timestamp.

If an older exchange-timestamped update arrives after a newer one for the same price level, it is discarded. This approach may lead to slight discrepancies between the local order book state and the actual state on the exchange.

</div>

<div class="alert alert-info">

**Note: Order Book Inconsistency Between Feeds**  
Each depth feed may present a different snapshot of the order book. When combining them, inconsistencies may arise even if the fused result reflects more up-to-date information.

For example:

- A depth feed may show a best bid/ask (BBO) at 10/11.
- A more current book ticker feed may show BBO at 10/14.

When fused:

The resulting book shows BBO at 10/14.
But price levels above 14 (e.g., 15 and higher) still reflect outdated data from the original depth feed. This results in a partially updated and inconsistent order book. To maintain consistency with the depth feed’s original intent, you would need to build and maintain a separate order book for each feed.

</div>

## Backtest Results Comparison

We now compare backtesting results between fused and non-fused data.

### Backtesting with Non-Fused Data

### Backtesting with Fused Data

You may notice slight differences in order fills, which lead to position discrepancies and, ultimately, equity differences — particularly between 03:00 and 15:00. These order fill differences are closely tied to the order placement behavior, which depends on the characteristics of the strategy. The differences in the BBO as shown above can result in significant equity divergence during backtesting.

<div class="alert alert-info">

**Note:** Some tutorial—especially older ones—were not backtested with fused market depth.

</div>

## Wrapping up

Since it uses more frequent data feeds, the backtesting process takes longer. There is always a trade-off between accuracy and speed in backtesting—there is no one-size-fits-all solution. Relaxing certain conditions, such as order queue position and latency modeling, can significantly speed up the process. Not all strategies require precise modeling of these factors, especially when dealing with small tick sizes and highly volatile assets like BTCUSDT. Please see [the next tutorial](https://hftbacktest.readthedocs.io/en/latest/tutorials/Accelerated%20Backtesting.html) about accelerated backtesting.